# Análisis de Eficiencia Espacial: Detour Factor (Factor de Desviación)

El **Detour Factor** (o coeficiente de circuidad) mide la eficiencia de una red de transporte comparando la distancia real recorrida a través de la infraestructura contra la distancia teórica en línea recta (Haversine).

**Fórmula:**
$$DF = \frac{Distancia_{Red} + Distancia_{Caminata}}{Distancia_{Haversine}}$$

Un valor de **1.0** indica una ruta perfectamente directa. Valores superiores a **1.5** suelen indicar ineficiencias topológicas, barreras urbanas o rodeos excesivos por transbordos.

En este notebook realizaremos:
1. Análisis estadístico de 500 viajes aleatorios.
2. Visualización de la trazabilidad y sistemas involucrados.
3. Mapeo interactivo de rutas específicas.

In [1]:
# ==========================================
# 2. Inicialización del Entorno y Librerías
# ==========================================
%load_ext autoreload
%autoreload 2
%matplotlib inline 

import sys
import os
import folium
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display


# Silenciar advertencias futuras para un reporte limpio
warnings.filterwarnings('ignore', category=FutureWarning)

# Asegurar que Jupyter encuentra la carpeta raíz del proyecto
proyecto_path = os.path.abspath('..')
if proyecto_path not in sys.path:
    sys.path.append(proyecto_path)
### Visualizador de utils
from src.core.utils.utils_detaur_factor import render_vft_detour_map
from src.infrastructure.go_client.client import fetch_full_network
from src.api.schemas.schemas import GeoJSONTransportSchema
from src.core.services.graph_builder import VFTGraphBuilder
from src.core.algorithms.topologicalIndicators.detaurFactor import DetourFactorOrchestrator
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'sans-serif'

In [3]:
# ==========================================
# 3. Descarga de Red y Construcción del Grafo
# ==========================================
print("📥 Descargando red de transporte...")
raw_data = await fetch_full_network()

print("⚙️ Construyendo Grafo Dirigido VFT...")
validated_payload = GeoJSONTransportSchema(**raw_data)

plt.ioff() 
GRAF= VFTGraphBuilder(validated_payload)
G = GRAF.build_graph()
plt.close('all')
plt.ion()

orchestrator = DetourFactorOrchestrator(G)
print(f"✅ Grafo construido: {G.number_of_nodes()} nodos topológicos.")

📥 Descargando red de transporte...
2026-05-31 22:42:41 | INFO     | VFT_Model | Construyendo el puente hacia el módulo espacial de Go...
2026-05-31 22:42:41 | INFO     | VFT_Model | Solicitando capa espacial a: http://localhost:8080/movilidad/mapas/geojsonEstacion
2026-05-31 22:42:41 | INFO     | VFT_Model | Solicitando capa espacial a: http://localhost:8080/movilidad/mapas/geojsonLinea
2026-05-31 22:42:42 | INFO     | VFT_Model | Red extraída: 22766 entidades espaciales listas para VFT.
⚙️ Construyendo Grafo Dirigido VFT...
2026-05-31 22:42:42 | INFO     | VFT_Model | Iniciando VFTGraphBuilder en modo: REALISTIC_INTEGRATION
2026-05-31 22:42:42 | INFO     | VFT_Model | Fase 1 y 2: Extrayendo nodos y trazos base...
2026-05-31 22:42:45 | INFO     | VFT_Model | Fase 2 completada: 64459 aristas interestación creadas.
2026-05-31 22:42:45 | INFO     | VFT_Model | Grafo Base construido: 11115 Nodos y 32103 Segmentos.
2026-05-31 22:42:45 | WARNING  | VFT_Model | Eliminadas 1543 aristas phantom

## 1. Muestreo Estadístico y Caso de Prueba Inicial
En esta sección generamos una muestra aleatoria de 100 viajes para entender el comportamiento base de la red y visualizamos el primer resultado obtenido sin ningún criterio de ordenamiento.

In [ ]:
# 1. Generar la muestra estadística global
tamano_muestra = 100
print(f"🎲 Generando muestra de {tamano_muestra} rutas...")
muestra_json = orchestrator.calculate_sample_routes(sample_size=tamano_muestra, seed=42, return_json=True)

if muestra_json:
    # Crear DataFrame de métricas
    df_metrics = pd.DataFrame([m['metrics'] for m in muestra_json])
    
    # Resumen estadístico
    print("\n📊 ESTADÍSTICAS GLOBALES DEL DETOUR FACTOR:")
    _df_desc = df_metrics['Factor_Desviacion'].describe().to_frame().T
    display(_df_desc)
    
    # Visualizar el primer caso de la lista
    primer_caso = muestra_json[0]
    mapa_ini, _, _, _, _, _ = render_vft_detour_map(
        primer_caso, 
        title=f"CASO INICIAL ALEATORIO (DF: {primer_caso['metrics']['Factor_Desviacion']})"
    )
    display(mapa_ini)

## 2. Análisis de Máxima Eficiencia: Las 5 Mejores Rutas
Aquí identificamos los trayectos donde la red de transporte es más competitiva frente al automóvil (trayectos casi directos). Visualizamos el mapa de la ruta con el menor Factor de Desviación.

In [ ]:
# 1. Mostrar tabla de las 5 mejores
print("✅ TOP 5 RUTAS MÁS EFICIENTES (Cercanas a 1.0):")
df_mejores = df_metrics.sort_values(by="Factor_Desviacion", ascending=True).head(5)
_cols_rutas = ['Origen', 'Destino', 'Sistemas_Involucrados', 'Distancia_Red_km', 'Factor_Desviacion']
display(df_mejores[_cols_rutas])

# 2. Identificar y graficar el mejor caso absoluto
mejor_caso = min(muestra_json, key=lambda x: x['metrics']['Factor_Desviacion'])
mapa_mejor, _, _, _, _, _ = render_vft_detour_map(
    mejor_caso, 
    title=f"LA RUTA MÁS EFICIENTE (DF: {mejor_caso['metrics']['Factor_Desviacion']})"
)
display(mapa_mejor)

## 3. Análisis de Puntos Críticos: Las 5 Peores Rutas
Identificamos los trayectos que presentan mayores ineficiencias, ya sea por falta de conexiones directas o rodeos excesivos. Mapeamos el "Peor Caso" para analizar visualmente la falla en la topología.

In [ ]:
# 1. Mostrar tabla de las 10 peores
print("🚨 TOP 10 RUTAS CON MAYOR INEFICIENCIA (Rodeos Críticos):")
df_peores = df_metrics.sort_values(by="Factor_Desviacion", ascending=False).head(10)
_cols_rutas = ['Origen', 'Destino', 'Sistemas_Involucrados', 'Distancia_Red_km', 'Factor_Desviacion']
display(df_peores[_cols_rutas])

# 2. Identificar y graficar el peor caso absoluto
peor_caso = max(muestra_json, key=lambda x: x['metrics']['Factor_Desviacion'])
mapa_peor, _, _, _, _, _ = render_vft_detour_map(
    peor_caso, 
    title=f"LA RUTA MÁS INEFICIENTE (DF: {peor_caso['metrics']['Factor_Desviacion']})"
)
display(mapa_peor)

## 4. Análisis de Ruta Arbitraria Personalizada (Caso de Estudio)
En esta sección evaluamos un viaje específico utilizando coordenadas exactas (Longitud, Latitud). Esto permite analizar la "primera y última milla" (caminata hacia la estación más cercana) y el factor de desviación para trayectos reales.

**Caso de Estudio:**
* **Origen:** Zona Sur, fuera del Anillo Periférico.
* **Destino:** Zona Oriente, inmediaciones del Autódromo Hermanos Rodríguez.

## 5. Galería de Casos de Estudio: Análisis de Trayectos Específicos
En esta sección, evaluamos una serie de rutas estratégicas seleccionadas por su relevancia en la conectividad de la Ciudad de México. 

Cada caso analiza:
1. **Contexto Geográfico:** Por qué son importantes estos puntos.
2. **Eficiencia Topológica:** Cómo se comporta la red frente a la distancia directa.
3. **Métrica de Caminata:** La distancia necesaria para conectar con la red formal (Primera y Última Milla).

In [ ]:
from IPython.display import display, Markdown

# 1. DEFINICIÓN DE RUTAS (Agrega aquí tantas como necesites)
casos_estudio = [
    {
        "titulo": "Caso 1: Periferia Extrema Sur hacia Centro-Oriente",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico hacia el Hub de entretenimiento del Autódromo. Evalúa la dificultad de salir de zonas con baja densidad de transporte formal.",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.09342206113304, 19.4059272938029)
    },
    {
        "titulo": "Caso 2: Conectividad Sur-Centro/Poniente (Milpa Alta a Reforma)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Sur) hacia el CBD Reforma.",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.17161535569772, 19.42527244485531) 
    },
    {
        "titulo": "Caso 3: Conectividad Sur-Poniente (Milpa Alta a Santa Fé)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Sur) hacia el CBD Santa Fé.",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.26937963016258 ,19.357049109027344) 
    },
    {
        "titulo": "Caso 4: Conectividad Sur-Poniente Norte (Milpa Alta al punto más cercano a FES Acatlán)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Sur) hacia Centro Educativo FES Acatlán.",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.2098132566566, 19.489900115538866) 
    },
    {
        "titulo": "Caso 5: Conectividad Sur-Centro/Sur (Milpa Alta a CU)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Sur) hacia el Centro Educativo Ciudad Universitaria.",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.184797668633, 19.33163949142085)
    },
    {
        "titulo": "Caso 6: Conectividad Sur-Centro/Sur (Milpa Alta a Coyoacán)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Sur) hacia Zona Residencial Centro Sur coyoacán.",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.15633349395276, 19.342445017191288)
    },
    {
        "titulo": "Caso 7: Conectividad Oriente-Centro/Sur (Santa Catarina a CU)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Oriente) hacia Centro Educativo Ciudad Universitaria.",
        "coords_o": (-98.96533517329291, 19.315314989332904),
        "coords_d": (-99.184797668633, 19.33163949142085) 
    },
    {
        "titulo": "Caso 8: Conectividad Oriente-Centro/Sur (Santa Catarina al Punto más cercano a FES Acatán)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Oriente) hacia Centro Educativo FES Acatlán.",
        "coords_o": (-98.96533517329291, 19.315314989332904),
        "coords_d": (-99.2098132566566, 19.489900115538866) 
    },
    {
        "titulo": "Caso 9: Conectividad Sur-Oriente (Milpa Alta - Tláhuac/Santa Catarina)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Sur) hacia zonas residenciales fuera del Anillo Periférico (Oriente).",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-98.9648158208182, 19.315197623324714) 
    },
    {
        "titulo": "Caso 10",
        "descripcion": "Milpa Alta - Ajusco",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.22739372393065, 19.280519282861828) 
    },
    {
        "titulo": "Caso 11",
        "descripcion": "Ajusco - Prepa 1",
        "coords_o": (-99.22739372393065, 19.280519282861828),
        "coords_d": (-99.12138370600817, 19.27230448210211) 
    },
    {
        "titulo": "Caso 11",
        "descripcion": "Parque de Los olivos - Centro de Tláhuac",
        "coords_o": (-99.00381778682763, 19.251950791794982),
        "coords_d": (-99.00476113482334, 19.27063980641322) 
    }
]

# 2. BUCLE OPTIMIZADO DE RENDERIZADO
for idx, caso in enumerate(casos_estudio, 1):
    display(Markdown(f"### {caso['titulo']}"))
    display(Markdown(f"**Contexto:** {caso['descripcion']}"))
    
    res = orchestrator.calculate_custom_route(caso['coords_o'], caso['coords_d'], return_json=True)
    
    if res:
        mapa, ini, fin, df_val, d_hav, d_red = render_vft_detour_map(
            res, 
            title=f"Visualización: {caso['titulo']}"
        )
        
        metricas_caminata = res['metrics'].get('Consideraciones_Reales', {})
        c1 = metricas_caminata.get('Caminata_1ra_Milla_Metros', 0)
        c2 = metricas_caminata.get('Caminata_Ultima_Milla_Metros', 0)
        
        resumen_md = f"""
| Métrica | Valor |
| :--- | :--- |
| **Punto de Abordaje** | {metricas_caminata.get('Estacion_Abordaje', 'N/A')} |
| **Punto de Descenso** | {metricas_caminata.get('Estacion_Descenso', 'N/A')} |
| **Factor de Desviación** | `{df_val}` |
| **Distancia en Red** | {d_red} km |
| **Tiempo Promedio en recorrer la Red (20 km/hr)** | {float(d_red)/20} hr |
| **Tiempo Promedio en recorrer la Distancia Geográfica (20 km/hr)** | {float(d_hav)/20} hr |
| **Distancia en Haversine** | {d_hav} km |
| **Caminata Total (1ra + Última)** | {round(c1 + c2, 2)} metros |
"""
        display(Markdown(resumen_md))
        display(mapa)
        display(Markdown("---"))
    else:
        display(Markdown(f"⚠️ *No se pudo calcular la ruta para: {caso['titulo']}*"))

In [8]:
# ==========================================
# EXPORTACIÓN QGIS — Mapa 3: Factor de Desviación
# ==========================================
from pathlib import Path
from shapely.geometry import Point
import geopandas as gpd
import pandas as pd

EXPORT_PATH = Path("../../Transport-gis-zmvm-mjg/data/processed/FactorDesviacion.gpkg")

# 1. Extraer puntos de origen con su DF desde muestra_json
puntos = []
for route in muestra_json:
    try:
        nodo_inicio = route['map_data']['network_route'][0]
        puntos.append({
            'DF':     float(route['metrics']['Factor_Desviacion']),
            'origen': str(route['metrics'].get('Origen', '')),
            'destino': str(route['metrics'].get('Destino', '')),
            'dist_red_km': float(route['metrics'].get('Distancia_Red_km', 0)),
            'geometry': Point(nodo_inicio['lon'], nodo_inicio['lat'])
        })
    except (KeyError, IndexError):
        continue

gdf_puntos = gpd.GeoDataFrame(puntos, crs="EPSG:4326").to_crs("EPSG:32614")
gdf_puntos.to_file(EXPORT_PATH, layer="df_puntos", driver="GPKG", engine="pyogrio")
print(f"✅ Capa puntos: {len(gdf_puntos)} rutas exportadas")

# 2. Coroplético por alcaldía (DF promedio)
from src.infrastructure.go_client.client_spatial import fetch_territorial_polygons
poligonos_data = await fetch_territorial_polygons(entidades=["Ciudad de México", "México"])
gdf_pol = gpd.GeoDataFrame.from_features(
    poligonos_data.get('features', poligonos_data.get('data', {}).get('features', [])),
    crs="EPSG:4326"
).to_crs("EPSG:32614")

if 'nombre' in gdf_pol.columns:
    gdf_pol = gdf_pol.dissolve(by='nombre').reset_index()

# Spatial join: asignar cada punto a su alcaldía
joined = gpd.sjoin(gdf_puntos, gdf_pol[['nombre', 'geometry']], how='left', predicate='within')
df_agg = joined.groupby('nombre')['DF'].agg(['mean', 'count']).reset_index()
df_agg.columns = ['nombre', 'DF_promedio', 'n_rutas']

gdf_coro = gdf_pol.merge(df_agg, on='nombre', how='left')
gdf_coro['nombre'] = gdf_coro['nombre'].astype(object)
gdf_coro = gdf_coro[['nombre', 'DF_promedio', 'n_rutas', 'geometry']]

gdf_coro.to_file(EXPORT_PATH, layer="df_por_alcaldia", driver="GPKG", engine="pyogrio")
print(f"✅ Capa coroplético: {len(gdf_coro)} demarcaciones")
print(f"\nTop 5 mayor DF promedio:")
print(gdf_coro.dropna().nlargest(5, 'DF_promedio')[['nombre','DF_promedio','n_rutas']].to_string(index=False))

✅ Capa puntos: 100 rutas exportadas
2026-05-31 22:43:39 | INFO     | VFT_Model | Solicitando polígonos territoriales a APIMETRO para: ['Ciudad de México', 'México']
2026-05-31 22:43:40 | INFO     | VFT_Model | Polígonos combinados exitosamente. Total features: 157
✅ Capa coroplético: 141 demarcaciones

Top 5 mayor DF promedio:
             nombre  DF_promedio  n_rutas
       Chimalhuacán     3.510000      1.0
            Tláhuac     3.340000      2.0
Ecatepec de Morelos     2.120000      1.0
          Iztacalco     1.783333      3.0
         Iztapalapa     1.684444      9.0
